# MyCON Reconstruction R3 — real 96-second room, safe COLMAP pilot

Run in a **private Google Colab** runtime with access to your own MyCon_R4 Drive folder. Reads original capture and historical 6999 artifacts **without modifying them**. The notebook creates a new `recon-*` run, hashes the historical video, selects new Train/Test frames, and optionally executes fresh COLMAP. No Gaussian training, resume, checkpoint writing to historical folders or baseline re-labeling is included.

The expected historical **upright-video** SHA256 and size are taken from existing R4 provenance; they MUST be independently verified here before use. Frames sampled from this video must **never** inherit old poses. New poses are unscaled COLMAP reconstruction units, not physical metres. Single-session temporal holdout is exploratory only. The separate 35-second video is NOT the input to this notebook.

Default: network install and fresh COLMAP are OFF; set the two opt-in flags explicitly when ready. The notebook does not start long GS training regardless of flags.

In [ ]:
from pathlib import Path
import os,sys,subprocess,hashlib,shutil,json,datetime
RUN_ID = "recon-room96-" + datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ")
INSTALL_COLMAP = False           # explicitly opt in to apt-get install
EXECUTE_FRESH_COLMAP = False     # explicitly opt in to real sparse SfM
GS_HEAVY_TRAINING_AUTHORIZED = False   # NEVER start training from this notebook
SOURCE_SHA256 = "2f58a3df2bfe955cb5935d748cf58b47cc1928fc12d6b0ea7bc5610aac5bff6a"
SOURCE_SIZE_BYTES = 651883372
CAPTURE_ORIGINAL_SHA256 = "b6f8e01a258e0f1dcaa98b536fa835099e1b43f872a5fd2acd1018693f078518"
def sha256_file(path):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        for chunk in iter(lambda:f.read(8*1024*1024),b""):h.update(chunk)
    return h.hexdigest()
print("RUN_ID",RUN_ID,"python",sys.version.split()[0],"heavy_training",GS_HEAVY_TRAINING_AUTHORIZED)
assert not GS_HEAVY_TRAINING_AUTHORIZED, "GS training requires a separate explicit authorization and workflow"


## Install source code into disposable Colab workspace (not inside Drive)

In [ ]:
REPO_DIR=Path("/content/mycon_reconstruction_code")
assert not REPO_DIR.exists(), "Refuse dirty/previous source checkout"
subprocess.run(["git","clone","--depth=1","--single-branch","--branch",
 "research/mycon-reconstruction-20261113","https://github.com/sayyedalimrj/MyCon.git",str(REPO_DIR)],check=True)
CODE_SHA=subprocess.check_output(["git","-C",str(REPO_DIR),"rev-parse","HEAD"],text=True).strip()
assert not subprocess.check_output(["git","-C",str(REPO_DIR),"status","--porcelain"],text=True).strip()
assert (REPO_DIR/"repository/reconstruction/workflow.py").is_file()
os.chdir(REPO_DIR/"repository")
sys.path.insert(0,str(Path.cwd()))
print("Checkout_SHA",CODE_SHA)


## Optional controlled dependencies. Never install a GS trainer automatically.

In [ ]:
if INSTALL_COLMAP:
    subprocess.run(["apt-get","update"],check=True)
    subprocess.run(["apt-get","install","-y","colmap"],check=True)
    subprocess.run([sys.executable,"-m","pip","install","--disable-pip-version-check",
                    "-r",str(REPO_DIR/"repository/requirements-research-reconstruction.txt")],check=True)
from reconstruction.preflight import audit
p=audit()
print(json.dumps(p,indent=2))
if EXECUTE_FRESH_COLMAP:
    assert shutil.which("colmap"), "COLMAP not installed; enable INSTALL_COLMAP first"


## Mount Drive READ-ONLY by convention: do not create or edit anything in the source directory.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
R4=Path("/content/drive/MyDrive/MyCon_R4")
VIDEO=R4/"input_cache/worksite_v480_934414c04ed4ef8a/full_video_upright.mp4"
assert VIDEO.is_file(), f"Missing frozen R4 source: {VIDEO}"
assert VIDEO.stat().st_size==SOURCE_SIZE_BYTES, "Historical video length mismatch"
actual_hash=sha256_file(VIDEO)
assert actual_hash==SOURCE_SHA256, "Historical full video SHA mismatch: STOP"
print("VERIFIED historical original full_video_upright",VIDEO.name,actual_hash)


## Verified LOCAL COPY of source video. The Drive source remains immutable.

In [ ]:
CACHE=Path("/content/mycon_r3_verified_source")
CACHE.mkdir(parents=True,exist_ok=False)
LOCAL_VIDEO=CACHE/"full_video_upright.mp4"
with VIDEO.open("rb") as source, LOCAL_VIDEO.open("xb") as dest:
    shutil.copyfileobj(source,dest,8*1024*1024)
assert LOCAL_VIDEO.stat().st_size==SOURCE_SIZE_BYTES
assert sha256_file(LOCAL_VIDEO)==SOURCE_SHA256
print("Local isolated VIDEO copy verified; Drive source untouched")


## Real frame selection with immutable train/test splits and SHA receipts.

In [ ]:
from reconstruction.workflow import prepare
RUN_ROOT=Path("/content/mycon_reconstruction_r3_runs")
RUN_ROOT.mkdir(parents=True,exist_ok=True)
RUN_DIR=RUN_ROOT/RUN_ID
assert not RUN_DIR.exists(), "Never rerun into an existing Run ID"
manifest=prepare(run_root=RUN_ROOT,run_id=RUN_ID,train_video=LOCAL_VIDEO,
    train_session="room96_verified_capture",test_video=None,test_session=None,
    sample_hz=2.0,max_frames=180,min_blur=35,min_mean=25,max_mean=230,
    max_clip=0.38,embargo_sec=1.0)
print("ACTUAL selected frame counts:",manifest["counts"])
print("Split:",manifest["train_test_protocol"])
assert manifest["source_videos"]["room96_verified_capture"]["source_sha256"]==SOURCE_SHA256


## Fresh COLMAP is OPTIONAL and train-only. Test views are localized after train mapper is frozen; no new GS training.

In [ ]:
from reconstruction.workflow import run_colmap,Blocked
colmap_result={"status":"not_executed_colmap_opt_in_required"}
if EXECUTE_FRESH_COLMAP:
    try:
        colmap_result=run_colmap(RUN_DIR,"colmap",execute=True)
    except Exception as exc:
        colmap_result={"status":"failed_preserve_logs_and_partial_run","error":str(exc)}
print(json.dumps(colmap_result,indent=2))


## Inspect coverage, train-only GS input readiness, runtime state and NO-TRAIN plan.

In [ ]:
from reconstruction.coverage import coverage
from reconstruction.workflow import train_plan,receipt
from reconstruction.gs_experiment import prepare_gs_dataset
if not (RUN_DIR/"coverage_report.json").exists():
    coverage_result=coverage(RUN_DIR)
else:
    coverage_result=json.loads((RUN_DIR/"coverage_report.json").read_text())
gs_dataset={"status":"not_executed_no_new_colmap"}
if colmap_result.get("status")=="colmap_executed":
    gs_dataset=prepare_gs_dataset(RUN_DIR) # COPY training images only; NO GS TRAIN
if not (RUN_DIR/"gs_training_plan.json").exists():
    train_plan(RUN_DIR)
p=audit(RUN_DIR)
with (RUN_DIR/"colab_preflight.json").open("x") as f:
    json.dump(dict(p,source_sha256=SOURCE_SHA256,code_sha=CODE_SHA,gs_heavy_training_authorized=False),f,indent=2)
print("Camera coverage status:",coverage_result["angular_coverage"]["state"])
print("GS dataset:",gs_dataset.get("status","prepared_only"),"HEAVY TRAINING NOT RUN")


## Save recovered artifacts in a NEW, separate Drive folder — copy-once, hash verification, never overwrite.

In [ ]:
receipt_result=receipt(RUN_DIR)
DRIVE_RUN_ROOT=R4/"research_reconstruction_r3_runs"
DRIVE_RUN_ROOT.mkdir(parents=True,exist_ok=True)
DRIVE_DEST=DRIVE_RUN_ROOT/RUN_ID
assert not DRIVE_DEST.exists(), "Existing Drive run! Never overwrite; use new RUN_ID"
shutil.copytree(RUN_DIR,DRIVE_DEST,symlinks=False)
for src in RUN_DIR.rglob("*"):
    if src.is_file():
        rel=src.relative_to(RUN_DIR); target=DRIVE_DEST/rel
        assert target.is_file() and sha256_file(src)==sha256_file(target),f"Drive sync integrity failure: {rel}"
print("CREATE-ONLY Drive snapshot verified:",DRIVE_DEST)
print("No new checkpoint/PLY is claimed. No historical file modified.")


## Recovery rules (manual): the previous Drive snapshot is immutable. A new attempt needs a NEW run ID. Heavy Gaussian training is intentionally absent.

In [ ]:
assert GS_HEAVY_TRAINING_AUTHORIZED is False
print("COLMAP executed:",colmap_result.get("status")=="colmap_executed")
print("GPU training: NOT_EXECUTED_UNAUTHORIZED")
print("If runtime disconnects, inspect the SHA-verified unique Drive snapshot; DO NOT rerun into it.")
print("Any future GS checkpoint must come from an approved NEW run and be copied create-only with independent SHA-256.")


## Future-only recovery: strict checkpoint copy gate
This function is defined but **not called**. No training or archival of the old 6999/recovery artifacts is authorized. It accepts only a newly created checkpoint inside this new run after separately documented explicit permission; never reuse a new run ID.

In [ ]:
from reconstruction.checkpoint_safety import preserve_new_checkpoint
FUTURE_HEAVY_TRAINING_APPROVAL=False
if FUTURE_HEAVY_TRAINING_APPROVAL:
    raise RuntimeError("No current heavy training approval; do not change without written authorization")
print("New-checkpoint archival helper loaded but NOT_EXECUTED.")
# After a SEPARATELY APPROVED fresh training run, an operator may call:
# preserve_new_checkpoint(run_dir=RUN_DIR,
#     checkpoint=RUN_DIR/"gs_training_NEW_OPT_IN"/"checkpoint_NEW.pt",
#     archive_root=DRIVE_RUN_ROOT, training_approved=True)
# Never pass any path from historical MyCon_R4/gsplat_result/recovery or ckpts.
